In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [2]:
data_ace_24 = pd.read_csv("../data/Ace_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_24 = pd.read_csv("../data/Discover_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_ace_af = pd.read_csv("../data/Ace_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_af = pd.read_csv("../data/Discover_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')

In [3]:
def build_models_adaptation():
    models = {
        "linreg": LinearRegression(),
        "boost": LGBMRegressor(
            n_estimators=1500,
            max_depth=3,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=42,
            verbose=-1),
        "pls": PLSRegression(n_components=100,
            scale=True,
            max_iter=2000)}
    return models

def adapt_ace_to_discover(ace_df, disc_df, future_lags, split_date="2022-01-01", adapt_model_name="linreg"):
    """
    Адаптация ace -> discover

    1. K обучается только на пересечении тренировочных данных ace и discover
    2. Ace до появления discover адаптируется в домен discover
    3. Возвращаются адаптированный ace_old_train и discr_train, disc_test
    """

    ace = ace_df.sort_index()
    disc = disc_df.sort_index()
    split_date = pd.Timestamp(split_date)

    ace_train = ace.loc[:split_date]
    disc_train = disc.loc[:split_date]
    disc_test = disc.loc[split_date:]

    overlap_idx = ace_train.index.intersection(disc_train.index)

    ace_overlap = ace_train.loc[overlap_idx]
    disc_overlap = disc_train.loc[overlap_idx]

    feature_cols = [c for c in ace.columns if c not in future_lags]

    sc_ace = StandardScaler().fit(ace_overlap[feature_cols])
    sc_disc = StandardScaler().fit(disc_overlap[feature_cols])

    X_ace = sc_ace.transform(ace_overlap[feature_cols])
    X_disc = sc_disc.transform(disc_overlap[feature_cols])

    # Модель адаптации K: ace -> discover
    models = build_models_adaptation()
    
    if adapt_model_name not in models:
        raise ValueError(f"Неизвестная модель: {adapt_model_name}. Доступны: {list(models.keys())}")
    
    K = models[adapt_model_name]

    # Используем, так как таргет многомерный
    if adapt_model_name == "boost":
        from sklearn.multioutput import MultiOutputRegressor
        K = MultiOutputRegressor(K)
        
    K.fit(X_ace, X_disc)

    # ace до появления discover
    ace_old = ace.loc[:disc.index.min()]
    ace_old = ace_old[feature_cols]

    X_ace_old = sc_ace.transform(ace_old)
    X_ace_old_adapted = K.predict(X_ace_old)
    X_ace_old_adapted = sc_disc.inverse_transform(X_ace_old_adapted)

    ace_old_adapted = pd.DataFrame(X_ace_old_adapted, index=ace_old.index, columns=feature_cols)

    for col in future_lags:
        ace_old_adapted[col] = ace_df.loc[ace_old.index, col]

    return ace_old_adapted, disc_train, disc_test, K, sc_ace, sc_disc

In [4]:
def build_models(random_state=42):
    models = {}
    models['Linear'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lin", LinearRegression())
    ])
    
    models['Ridge'] = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(
            alpha=1.0,
            solver='auto',
            random_state=random_state))
    ])

    models['Lasso'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(
            alpha=0.0005,
            tol=0.01,
            max_iter=500, 
            random_state=random_state))
    ])
    
    models['LGBM'] = Pipeline([
        ("boost", LGBMRegressor(
            n_estimators=1500,
            max_depth=3,
            early_stopping_rounds=50,
            random_state=random_state,
            verbose=-1))
    ])
    
    models['MLP'] = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(256, 256, 128),
            solver='adam',
            alpha=1e-4,
            learning_rate_init=1e-3,
            early_stopping=True,
            validation_fraction=0.1,
            max_iter=1500,
            random_state=random_state))
    ])
    return models
    
models = build_models()

def evaluate_M_D(ace_old_adapted, disc_train, disc_test, split_date, target, future_lags, adaptation_method='linreg', delays='24h', results_list=None):
    """
    Обучение M_D:
    - train = адаптированный старый ace + discover_train
    - test = discover_test
    """
    train_adapt = pd.concat([ace_old_adapted, disc_train], axis=0).sort_index()

    feature_cols = [c for c in train_adapt.columns if c not in future_lags]

    X_train = train_adapt[feature_cols].values
    y_train = train_adapt[target].values

    X_test = disc_test[feature_cols].values
    y_test = disc_test[target].values
    
    results = {}
    results[target] = {}

    for name, model in models.items():
        if name == 'LGBM':
            # Для бустинга выделяем валидационный набор
            X_tr, X_val, y_tr, y_val = train_test_split(
                X_train, y_train, test_size=0.1, random_state=42)

            model.fit(X_tr, y_tr,
                boost__eval_set=[(X_val, y_val)],
                boost__eval_metric='l2')
        else:
            model.fit(X_train, y_train)
            
        y_pred = model.predict(X_test)

        if results_list is None:
            results_list = []
            
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results_list.append({
            'target': target,
            'delays': delays,
            'adaptation_method': adaptation_method,
            'forecast_model': name,
            'RMSE': rmse,
            'MAE': mae,
            'R2': r2
        })

        print(f"{name}: rmse={rmse:.4f}, mae={mae:.4f}, r2={r2:.4f}")
        
    return results

In [5]:
# 0. Копирование загруженных данных в отдельные переменные
data_ace_24_copy = data_ace_24.copy()
data_discover_24_copy = data_discover_24.copy()
data_ace_af_copy = data_ace_af.copy()
data_discover_af_copy = data_discover_af.copy()

# 1. Задание переменных для адаптации
split_date = "2022-01-01"
future_lags = [f'Dst_plus{i}' for i in range(1, 25)] # все сдвиги во времени вперед (задаются при обработке данных)
targets = ['Dst_plus1', 'Dst_plus2', 'Dst_plus3', 'Dst_plus6'] # таргеты, на которые делаем прогнозы

In [ ]:
# 2.1. Доменная адаптация с помощью моделей линейной регрессии (linreg), градиентного бустинга (boost) и метода проекций на латентные структуры (pls)
ace_adapted_lin_24, disc_train_24, disc_test_24, K_lin_24, sd_lin_24, sa_lin_24 = adapt_ace_to_discover(
    data_ace_24_copy,
    data_discover_24_copy,
    future_lags,
    split_date,
    adapt_model_name="linreg"
)

ace_adapted_gbr_24, disc_train_24, disc_test_24, K_gbr_24, sd_gbr_24, sa_gbr_24 = adapt_ace_to_discover(
    data_ace_24_copy,
    data_discover_24_copy,
    future_lags,
    split_date,
    adapt_model_name="boost"
)

ace_adapted_pls_24, disc_train_24, disc_test_24, K_pls_24, sd_pls_24, sa_pls_24 = adapt_ace_to_discover(
    data_ace_24_copy,
    data_discover_24_copy,
    future_lags,
    split_date,
    adapt_model_name="pls"
)

In [6]:
ace_adapted_lin_af, disc_train_af, disc_test_af, K_lin_af, sd_lin_af, sa_lin_af = adapt_ace_to_discover(
    data_ace_af_copy,
    data_discover_af_copy,
    future_lags,
    split_date,
    adapt_model_name="linreg"
)

ace_adapted_gbr_af, disc_train_af, disc_test_af, K_gbr_af, sd_gbr_af, sa_gbr_af = adapt_ace_to_discover(
    data_ace_af_copy,
    data_discover_af_copy,
    future_lags,
    split_date,
    adapt_model_name="boost"
)

ace_adapted_pls_af, disc_train_af, disc_test_af, K_pls_af, sd_pls_af, sa_pls_af = adapt_ace_to_discover(
    data_ace_af_copy,
    data_discover_af_copy,
    future_lags,
    split_date,
    adapt_model_name="pls"
)

In [8]:
results_list = []

In [ ]:
print(f"\n==== Depth - 24h ====")

print(f"\n==== Adaptation - linreg ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_D(ace_adapted_lin_24, disc_train_24, disc_test_24, split_date, target_col, future_lags, adaptation_method='linreg', delays='24h', results_list=results_list)

print(f"\n==== Adaptation - lgbm ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_D(ace_adapted_gbr_24, disc_train_24, disc_test_24, split_date, target_col, future_lags, adaptation_method='lgbm', delays='24h', results_list=results_list)

print(f"\n==== Adaptation - pls ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_D(ace_adapted_pls_24, disc_train_24, disc_test_24, split_date, target_col, future_lags, adaptation_method='pls', delays='24h', results_list=results_list)

In [9]:
print(f"\n==== Depth - autocorrelation function ====")

print(f"\n==== Adaptation - linreg ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_D(ace_adapted_lin_af, disc_train_af, disc_test_af, split_date, target_col, future_lags, adaptation_method='linreg', delays='auto_func', results_list=results_list)

print(f"\n==== Adaptation - lgbm ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_D(ace_adapted_gbr_af, disc_train_af, disc_test_af, split_date, target_col, future_lags, adaptation_method='lgbm', delays='auto_func', results_list=results_list)

print(f"\n==== Adaptation - pls ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_D(ace_adapted_pls_af, disc_train_af, disc_test_af, split_date, target_col, future_lags, adaptation_method='pls', delays='auto_func', results_list=results_list)


==== Depth - autocorrelation function ====

==== Adaptation - linreg ====

==== Forecast of DST_PLUS1 ====
Linear: rmse=3.4066, mae=2.4379, r2=0.9664
Ridge: rmse=3.4065, mae=2.4378, r2=0.9664
Lasso: rmse=3.4050, mae=2.4346, r2=0.9665
LGBM: rmse=3.5422, mae=2.4367, r2=0.9637
MLP: rmse=3.5339, mae=2.5543, r2=0.9639

==== Forecast of DST_PLUS2 ====
Linear: rmse=5.3410, mae=3.8696, r2=0.9175
Ridge: rmse=5.3409, mae=3.8695, r2=0.9175
Lasso: rmse=5.3368, mae=3.8671, r2=0.9176
LGBM: rmse=5.1734, mae=3.7003, r2=0.9226
MLP: rmse=5.7269, mae=4.1675, r2=0.9051

==== Forecast of DST_PLUS3 ====
Linear: rmse=6.8284, mae=4.9402, r2=0.8651
Ridge: rmse=6.8284, mae=4.9401, r2=0.8651
Lasso: rmse=6.8251, mae=4.9375, r2=0.8652
LGBM: rmse=6.7259, mae=4.7286, r2=0.8691
MLP: rmse=7.7874, mae=5.6117, r2=0.8245

==== Forecast of DST_PLUS6 ====
Linear: rmse=10.0071, mae=7.0244, r2=0.7103
Ridge: rmse=10.0071, mae=7.0244, r2=0.7103
Lasso: rmse=10.0058, mae=7.0224, r2=0.7104
LGBM: rmse=10.1220, mae=6.9097, r2=0.70

In [ ]:
results_df = pd.DataFrame(results_list)

In [ ]:
results_df

,target,delays,adaptation_method,forecast_model,RMSE,MAE,R2
0,Dst_plus1,auto_func,linreg,Linear,3.406553,2.437885,0.966428
1,Dst_plus1,auto_func,linreg,Ridge,3.406465,2.437825,0.966429
2,Dst_plus1,auto_func,linreg,Lasso,3.405002,2.434604,0.966458
3,Dst_plus1,auto_func,linreg,LGBM,3.542237,2.436744,0.963700
4,Dst_plus1,auto_func,linreg,MLP,3.533871,2.554266,0.963871
5,Dst_plus2,auto_func,linreg,Linear,5.341008,3.869577,0.917469
6,Dst_plus2,auto_func,linreg,Ridge,5.340924,3.869536,0.917471
7,Dst_plus2,auto_func,linreg,Lasso,5.336776,3.867084,0.917599
8,Dst_plus2,auto_func,linreg,LGBM,5.173448,3.700304,0.922566
9,Dst_plus2,auto_func,linreg,MLP,5.726872,4.167538,0.905113


In [12]:
results_df.to_excel("../results/models-adaptation-ace-to-discover_2022-01-01.xlsx", index=False)